# DriveMind ML track -- base measurement, QLoRA fine-tune, fine-tuned measurement

Three numbers, one environment, in that order. A base figure from one
runtime and a tuned figure from another are not comparable, so both are
produced here on the same card.

**Before running:** Settings -> Accelerator = `GPU T4 x2`, and Internet =
`On` (needed for pip and the model download). Session cap is 12 h; the
whole sequence is an estimated 4-6 h, unmeasured.

**Nothing here scores anything.** Every metric comes from
`ml.eval_holdout`, which gets it from `backend.services.ai_holdout`. This
file is a sequencer with a stopwatch -- a notebook that recomputed a
metric would be a second scorer that can disagree with the first.

**The order is the point.** Each step is cheaper than the next and can
refuse, so a stale corpus costs three minutes instead of five hours:

| Step | Cost | What it refuses on |
|---|---|---|
| deps | ~2 min | pip fails rather than replacing Kaggle's torch |
| corpus | ~1 min | `build_dataset` will not write a bad corpus |
| reference | ~10 s | split verification, on Linux, no GPU |
| dry run | ~2 min | stale policy stamp, over-length row (tokenizer only) |
| smoke | ~3 min | model does not load or emits no JSON |
| base eval | ~45 min | held-out rows are not the published ones |
| train | ~2-4 h | |
| tuned eval | ~45 min | |

When it finishes, paste `artifacts/base_holdout.txt` and
`artifacts/tuned_holdout.txt` back into the session that wrote this.

In [ ]:
# --- Cell 1: what this is running on -----------------------------------
#
# Reported, not logged, because a held-out figure is only comparable with
# another if the runtime is known. 4-bit on one card and bf16 on another
# are different measurements of the same weights.

import os
import subprocess
import sys
from pathlib import Path

MODEL = "Qwen/Qwen3-4B"

# One card, deliberately. A 4B model in 4-bit NF4 is ~2.5 GB and fits a
# 16 GB T4 with room to spare; `device_map="auto"` across two cards puts
# Trainer on its model-parallel path and shards a gradient-checkpointed
# PEFT model for no gain. The second T4 stays idle, and all three
# measurements come from the same device.
CUDA_DEVICE = "0"

print(f"python          {sys.version.split()[0]}")

assert sys.version_info >= (3, 10), (
    "backend/ uses zip(strict=True), so 3.10 is the hard floor. "
    "pyproject declares >=3.12, which is the tested floor, not this one."
)

try:
    import torch

except ImportError:
    raise SystemExit("torch is not installed -- is the accelerator set to GPU?")

print(f"torch           {torch.__version__}")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device visible. Settings -> Accelerator -> GPU T4 x2."
    )

for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)

    print(
        f"device {index}        {properties.name}, "
        f"{properties.total_memory // (1024 * 1024)} MiB, "
        f"capability {properties.major}.{properties.minor}, "
        f"CUDA {torch.version.cuda}"
    )

print(f"bf16 supported  {torch.cuda.is_bf16_supported()}")
print()
print("  T4 is Turing (7.5) and has no bfloat16. pick_dtype() selects")
print("  fp16 there; passing bf16=True on a T4 raises rather than")
print("  falling back, which is why that is decided at runtime.")


def sh(*command, cwd=None, env=None, check=True):
    """
    Run a command, stream its output into the notebook, return its code.

    No output capture: these steps run for tens of minutes and a
    progress bar you cannot see is indistinguishable from a hang.
    """

    print(f"\n$ {' '.join(str(part) for part in command)}\n", flush=True)

    completed = subprocess.run(list(command), cwd=cwd, env=env)

    if check and completed.returncode != 0:
        raise SystemExit(
            f"exit {completed.returncode} -- stopping here rather than "
            "running the next step on a failed one"
        )

    return completed.returncode

In [ ]:
# --- Cell 2: find the repository ----------------------------------------
#
# Two ways in, checked in this order:
#
#   1. This file's own checkout, when run as `python ml/kaggle_qlora.py`
#      on a machine that already has the repo. `__file__` is undefined in
#      a notebook cell, so on Kaggle this candidate simply does not exist.
#   2. A Kaggle Dataset containing the repo (Add Input -> Datasets). No
#      push, no token, no credentials. This is the path to use while the
#      ML track is unpushed.
#   3. `git clone REPO_URL`, for when it is pushed and public.
#
# There is deliberately no fourth way involving a personal access token.
# A notebook is a shared artifact and a token pasted into one is a
# committed secret.

REPO_URL = "https://github.com/Xtremephenom/DriveMind.git"

# Every file the sequence actually needs. Checked up front because the
# useful failure is "the ML track is not in this checkout", not
# `ModuleNotFoundError: ml` forty minutes later.
REQUIRED = (
    "backend/services/dataset/build.py",
    "backend/services/ai_holdout.py",
    "ml/hf_runner.py",
    "ml/eval_holdout.py",
    "ml/train_qlora.py",
)

ON_KAGGLE = Path("/kaggle/working").exists()
WORKDIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()


def is_checkout(path: Path) -> bool:
    return (path / "backend" / "services" / "dataset" / "build.py").exists()


def find_repo() -> Path:
    local = []

    if "__file__" in globals():
        local.append(Path(__file__).resolve().parent.parent)

    local.append(Path.cwd())

    for candidate in local:
        if is_checkout(candidate):
            print(f"  using the checkout at {candidate}")
            return candidate

    # /kaggle/input/<dataset>/ or /kaggle/input/<dataset>/<subdir>/,
    # depending on whether the zip had a top-level folder.
    uploaded = sorted(Path("/kaggle/input").glob("*/")) + sorted(
        Path("/kaggle/input").glob("*/*/")
    )

    for candidate in uploaded:
        if not is_checkout(candidate):
            continue

        print(f"  found an uploaded checkout at {candidate}")

        # /kaggle/input is read-only and the corpus build writes
        # data/*.jsonl, so it is copied out rather than used in place.
        # An existing copy is reused; delete WORKDIR/DriveMind by hand
        # after swapping the attached dataset for a newer one.
        import shutil

        destination = WORKDIR / "DriveMind"

        if not destination.exists():
            shutil.copytree(candidate, destination)
            print(f"  copied to {destination} (input is read-only)")

        return destination

    destination = WORKDIR / "DriveMind"

    if not is_checkout(destination):
        print("  no local or uploaded checkout found; cloning")
        sh("git", "clone", "--depth", "1", REPO_URL, str(destination))

    return destination


REPO = find_repo()

missing = [name for name in REQUIRED if not (REPO / name).exists()]

if missing:
    raise SystemExit(
        "This checkout is incomplete:\n  "
        + "\n  ".join(missing)
        + "\n\nThe ML track is not in origin/main yet. Either push it, or "
        "zip the working tree, upload it as a Kaggle Dataset, and attach "
        "it with Add Input."
    )

os.chdir(REPO)
sys.path.insert(0, str(REPO))

ARTIFACTS = REPO / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

print(f"\n  repo            {REPO}")
print(f"  artifacts       {ARTIFACTS}")

if (REPO / ".git").exists():
    sh("git", "log", "-1", "--format=  commit          %h %s", cwd=REPO)

else:
    print("  commit          unknown (uploaded as files, not a checkout)")

In [ ]:
# --- Cell 3: dependencies, with torch protected -------------------------
#
# Kaggle's torch is paired with the image's CUDA runtime and driver.
# Letting pip resolve a fresh one is a multi-gigabyte download that can
# also break that pairing, and it does it quietly -- the install
# succeeds and the failure shows up later as a CUDA error.
#
# So torch goes into a constraints file pinned to the *exact* installed
# version, local label included (`2.9.0+cu126`). If the other four can
# be satisfied against it, they install and torch is untouched. If they
# cannot, pip cannot find that version on PyPI either and fails loudly.
# A loud failure here is the correct outcome: it means the pin set and
# this image disagree, which is a decision for a person.

import importlib.metadata
import tempfile

PINS = (
    "transformers==5.16.1",
    "peft==0.20.0",
    "accelerate==1.14.0",
    "bitsandbytes==0.50.2",
)

torch_before = torch.__version__

constraints = Path(tempfile.gettempdir()) / "drivemind-constraints.txt"
constraints.write_text(f"torch=={torch_before}\n", encoding="utf-8")

print(f"  holding         torch=={torch_before}")

sh(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--constraint",
    str(constraints),
    *PINS,
)

print()

for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
    try:
        print(f"  {name:<16}{importlib.metadata.version(name)}")

    except importlib.metadata.PackageNotFoundError:
        print(f"  {name:<16}NOT INSTALLED")

torch_after = importlib.metadata.version("torch")

if torch_after != torch_before:
    raise SystemExit(
        f"torch changed from {torch_before} to {torch_after} despite the "
        "constraint. Stop: the CUDA pairing is no longer the image's, and "
        "any number measured now is measured on an unknown runtime."
    )

print("\n  torch unchanged. The pins were resolved against CPython 3.14.2")
print("  win_amd64; if a linux wheel for one of them is missing, the pip")
print("  step above is where you find out.")

In [ ]:
# --- Cell 4: rebuild the corpus, and check it against Windows -----------
#
# `data/` is gitignored, so there are no rows here until this runs. The
# build is deterministic from seed 42 and refuses to write on duplicate
# case_id, cross-file overlap, a missing FileCategory, a signal outside
# the production vocabulary, or a prompt containing the answer.
#
# The hashes below were measured on the Windows machine that built the
# corpus the dataset card describes. Comparing against them turns "the
# corpus should be reproducible on Linux" into a measurement.
#
# They are **LF-normalized**. `writer.write_jsonl` opens with the default
# newline translation, so the same rows are CRLF on Windows and LF on
# Linux: raw bytes differ for a reason that has nothing to do with the
# corpus. Normalizing isolates a real content difference from a line
# ending, which is the only comparison worth making across platforms.
#
# A mismatch does not invalidate the run -- `evaluate_emitter`
# regenerates and verifies in place, so the scoring is self-consistent
# either way. It would mean the distributions in the dataset card were
# measured on a different corpus than the one being scored, which is
# worth knowing before a number is published.

import hashlib

WINDOWS_SHA256_LF = {
    "train": "103ea0face08040947f053b1aecd326b2d8baed1369eb7334d2ab612dc867630",
    "validation": "8f051377ee146db22060015a27bd388bfa4b40308d3b3bfbdf1ae0ac13b4e739",
    "test": "3ddb91f5a95b6fbf63d765ccf9b8f1e7153a599176333a00389ce379c41366ca",
    "gold": "2882da3182d2f480ea1d65cdb6f22f7d9711f6b37b1fc209b144f2d81a125d70",
    "red_team": "215b455196ef40b03934af70488d144236242a7b5b64547487b9d700f3693e3e",
}

from backend.services.dataset.build import build_dataset
from backend.services.decision.engine import POLICY_VERSION

print(f"  policy version  {POLICY_VERSION}\n")

summary = build_dataset()

print()

divergent = []

for name, expected in WINDOWS_SHA256_LF.items():
    raw = (REPO / "data" / f"{name}.jsonl").read_bytes()
    normalized = raw.replace(b"\r\n", b"\n")
    actual = hashlib.sha256(normalized).hexdigest()

    rows = len(normalized.splitlines())
    verdict = "match" if actual == expected else "DIVERGES from Windows"

    if actual != expected:
        divergent.append(name)

    print(f"  {name:<12}{rows:>6} rows  {actual[:16]}  {verdict}")

if divergent:
    print(
        "\n  WARNING: "
        + ", ".join(divergent)
        + " differ in content from the Windows build.\n"
        "  The run below is still internally consistent, but the dataset\n"
        "  card's measured distributions describe the other corpus."
    )

else:
    print("\n  All five match the Windows build, content-for-content.")

In [ ]:
# --- Cell 5: the deterministic reference, no GPU -------------------------
#
# `RuleBasedAIProvider` is a mirror of the engine, so this is 100/100/100
# with 0 unsafe escalations by construction. It is run anyway, and first,
# for two reasons: it proves the held-out verification works on this
# machine before any GPU time is spent, and it is the reference row the
# model's numbers get read against. A model figure with nothing beside it
# is hard to interpret; the same figure next to a known ceiling is not.
#
# Takes seconds. If it fails, nothing below is worth starting.

GPU_ENV = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": CUDA_DEVICE,
    "PYTHONPATH": str(REPO),
    # A tokenizers fork warning on every one of 1,126 generations is
    # noise that hides the real output.
    "TOKENIZERS_PARALLELISM": "false",
}

sh(
    sys.executable,
    "-m",
    "backend.services.ai_holdout",
    cwd=REPO,
    env=GPU_ENV,
)

In [ ]:
# --- Cell 6: the training data path, tokenizer only ----------------------
#
# Encodes all 9,000 rows and stops. No GPU, no base weights -- just the
# tokenizer, which is a few megabytes.
#
# This is the step that catches a stale policy stamp and a row over the
# length budget. Both are refusals rather than warnings, and both would
# otherwise surface after the 4-bit weights are resident: on a
# session-limited runtime, finding out now is worth the two minutes.
#
# Read the reported token distribution. `max_length` is 1,024 and the
# encoder raises rather than truncating, because the tail of a long
# prompt is where `signals` lives -- the part of the evidence the label
# most depends on. A truncated example trains the model to answer a
# question it was not shown.

sh(
    sys.executable,
    "-m",
    "ml.train_qlora",
    "--model",
    MODEL,
    "--dry-run",
    cwd=REPO,
    env=GPU_ENV,
)

In [ ]:
# --- Cell 7: does the model load and emit the contract? -----------------
#
# 12 hand-built cases, about three minutes once the weights are cached.
# Not a measurement: a model can pass all twelve by pattern matching on
# `category`, and the runner prints that caveat itself.
#
# It answers the one question worth answering before committing 45
# minutes -- does this model load in 4-bit on a T4, and does it emit
# something `parse_ai_response` accepts. The first run also downloads the
# base weights (~8 GB), so expect this cell to be dominated by that.

sh(
    sys.executable,
    "-m",
    "ml.eval_holdout",
    "--model",
    MODEL,
    "--smoke",
    cwd=REPO,
    env=GPU_ENV,
)

In [ ]:
# --- Cell 8: the base model, held out ------------------------------------
#
# 1,126 cases -- 1,000 test, 100 gold, 26 red_team -- generated one at a
# time, greedy. Estimated 45 minutes, unmeasured.
#
# This has to happen *before* training, and in this environment. A base
# figure taken from a different runtime is not a comparison, and a base
# figure taken after the adapter exists is a figure nobody will trust.
#
# `evaluate_emitter` regenerates every case and checks it against the row
# on disk before it scores anything. There is no --limit: a shortened run
# would either break that proof or bypass it, and then print under a
# heading that says "held-out".

sh(
    sys.executable,
    "-m",
    "ml.eval_holdout",
    "--model",
    MODEL,
    "--out",
    str(ARTIFACTS / "base_holdout.txt"),
    cwd=REPO,
    env=GPU_ENV,
)

In [ ]:
# --- Cell 9: QLoRA fine-tune ---------------------------------------------
#
# 8,000 rows at an effective batch of 16 is 500 optimizer steps for one
# epoch. Estimated 2-4 hours, unmeasured.
#
# `--eval-steps 125` rather than the default 50, and the reason is
# arithmetic: a validation pass is 1,000 rows at batch 2, so 500 forward
# passes, against 4,000 for the whole epoch of training. At every 50
# steps that is ten passes -- 5,000 forward passes spent on checkpoint
# selection, more than the training itself. Four passes still gives
# `load_best_model_at_end` something to choose between.
#
# Selection is on `eval_loss` over validation.jsonl, which is a proxy and
# is labelled as one: lower validation loss is not higher action
# agreement. Measuring agreement needs generation, and generating over
# 1,000 cases at every checkpoint costs more than the fine-tune. The
# metrics that matter are measured once, in the next cell.
#
# `test.jsonl` is not opened by this script. A checkpoint chosen by test
# performance has spent the test set.

sh(
    sys.executable,
    "-m",
    "ml.train_qlora",
    "--model",
    MODEL,
    "--out",
    str(ARTIFACTS / "qlora"),
    "--eval-steps",
    "125",
    cwd=REPO,
    env=GPU_ENV,
)

In [ ]:
# --- Cell 10: the fine-tuned model, held out -----------------------------
#
# Same command as cell 8 plus `--adapter`, same card, same corpus, same
# decoding. That is the whole point of running it this way: the only
# thing that differs between the two reports is the adapter.

sh(
    sys.executable,
    "-m",
    "ml.eval_holdout",
    "--model",
    MODEL,
    "--adapter",
    str(ARTIFACTS / "qlora"),
    "--out",
    str(ARTIFACTS / "tuned_holdout.txt"),
    cwd=REPO,
    env=GPU_ENV,
)

In [ ]:
# --- Cell 11: both reports, and how to read them -------------------------
#
# Printed together so the pair travels as one artifact. Everything in
# `artifacts/` under /kaggle/working is saved as notebook output; the
# adapter is a few hundred megabytes and `drivemind_run.json` sits next
# to it recording which policy's labels it learned and which base model
# it belongs on top of. An adapter without that file says neither.

for name in ("base_holdout.txt", "tuned_holdout.txt"):
    path = ARTIFACTS / name

    print("=" * 72)
    print(name)
    print("=" * 72)
    print(path.read_text(encoding="utf-8") if path.exists() else "  not written")
    print()

run_json = ARTIFACTS / "qlora" / "drivemind_run.json"

if run_json.exists():
    print("=" * 72)
    print("drivemind_run.json")
    print("=" * 72)
    print(run_json.read_text(encoding="utf-8"))

print("=" * 72)
print("artifacts")
print("=" * 72)

for path in sorted(ARTIFACTS.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACTS)!s:<44}{path.stat().st_size:>12,} B")

print()
print("  Read unsafe escalation rate first, not action agreement.")
print("  It is the primary safety metric and it is counted BEFORE the")
print("  gate clamps, so it measures what the model attempted rather")
print("  than what a user would have seen. A fine-tune that raises")
print("  agreement while attempting more escalations is worse, not")
print("  better, and reading agreement first would hide that.")
print()
print("  Structured-output validity is the second thing to read. A model")
print("  that cannot hold the {action, risk, explanation} contract has")
print("  no agreement figure worth discussing -- an unparseable response")
print("  lands the user on the deterministic recommendation, which is")
print("  safe and also means the model contributed nothing.")
print()
print("  Both reports and drivemind_run.json are what to paste back.")